# SI10-2026 | Ponderada | Análise de Sensibilidade em Métricas de Interface Digital

Nesta atividade, você vai analisar quais variáveis de uma interface digital têm maior impacto sobre a taxa de conversão.

A entrega deve ser feita neste notebook, com código, tabelas, gráficos e respostas curtas.

## Contexto

Uma equipe de produto quer decidir qual métrica de interface deve receber prioridade no próximo ciclo de melhoria.

Os dados representam observações diárias de um aplicativo de compras.

A métrica alvo é a taxa de conversão.

As variáveis de entrada são taxa de abandono do carrinho, profundidade média de scroll e tempo até o primeiro clique em produto.

## Preparação

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

# Definidas na preparação para estarem disponíveis antes de qualquer gráfico.
features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
]
target = "taxa_conversao_pct"

pd.set_option("display.precision", 3)


## Dados

Execute a célula abaixo para criar a base da atividade.

In [2]:
rng = np.random.default_rng(42)
n_dias = 180

taxa_abandono = rng.normal(48, 8, n_dias).clip(25, 75)
profundidade_scroll = rng.normal(62, 12, n_dias).clip(25, 95)
tempo_primeiro_clique = rng.normal(7, 2.2, n_dias).clip(2, 15)

ruido = rng.normal(0, 0.35, n_dias)
taxa_conversao = (
    7.5
    - 0.055 * taxa_abandono
    + 0.026 * profundidade_scroll
    - 0.085 * tempo_primeiro_clique
    + ruido
).clip(0.5, 9.0)

df = pd.DataFrame({
    "data": pd.date_range("2026-01-01", periods=n_dias, freq="D"),
    "taxa_abandono_carrinho_pct": taxa_abandono,
    "profundidade_scroll_pct": profundidade_scroll,
    "tempo_primeiro_clique_s": tempo_primeiro_clique,
    "taxa_conversao_pct": taxa_conversao,
})

df.head()

,data,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
0,2026-01-01,50.438,77.672,6.664,5.589
1,2026-01-02,39.680,64.633,7.843,6.042
2,2026-01-03,54.004,57.069,9.200,5.318
3,2026-01-04,55.525,75.275,4.671,5.944
4,2026-01-05,32.392,67.145,6.725,6.804


In [3]:
# Validação do esquema da base antes das análises.
colunas_esperadas = features + [target]
colunas_ausentes = [coluna for coluna in colunas_esperadas if coluna not in df.columns]

if colunas_ausentes:
    raise ValueError(f"Colunas ausentes na base: {colunas_ausentes}")


## Parte 1: Exploração

Crie ao menos um gráfico ou tabela para investigar a relação entre as variáveis de entrada e a taxa de conversão.

In [4]:
# Tabela e mapa de calor das correlações lineares.
correlacoes = df[features + [target]].corr()
display(correlacoes.round(3))

fig_correlacao = px.imshow(
    correlacoes,
    text_auto=".3f",
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    title="Correlação entre métricas de interface e taxa de conversão",
    labels={"color": "correlação"},
)
fig_correlacao.update_layout(height=550)
fig_correlacao.show()


,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
taxa_abandono_carrinho_pct,1.000,-0.068,-0.116,-0.643
profundidade_scroll_pct,-0.068,1.000,0.050,0.485
tempo_primeiro_clique_s,-0.116,0.050,1.000,-0.229
taxa_conversao_pct,-0.643,0.485,-0.229,1.000


In [5]:
# Variável com maior correlação absoluta com a taxa de conversão.
variavel_x = "taxa_abandono_carrinho_pct"

if variavel_x not in features:
    raise ValueError("Preencha variavel_x com uma variável da lista features.")

fig = px.scatter(
    df,
    x=variavel_x,
    y=target,
    opacity=0.70,
    title="Abandono do carrinho versus taxa de conversão",
    labels={
        variavel_x: "Taxa de abandono do carrinho (%)",
        target: "Taxa de conversão (%)",
    },
)
fig.show()


Escreva quais duas variáveis você escolheu para a análise de sensibilidade e justifique com evidências da exploração.

**Resposta:**

Escolhi **taxa de abandono do carrinho** e **profundidade de scroll**. Elas apresentam as duas maiores correlações, em valor absoluto, com a taxa de conversão: **-0,643** e **+0,485**; o tempo até o primeiro clique tem correlação menor, de **-0,229**. Assim, a exploração sugere o ranking abandono > scroll > primeiro clique.

Os sinais também orientam a decisão: maior abandono está associado a menor conversão; maior profundidade de scroll, a maior conversão; e maior demora até o primeiro clique, a menor conversão. Essas relações são associações observadas na base, não evidência causal.


## Parte 2: Modelo

Ajuste o modelo abaixo para estimar a taxa de conversão a partir das variáveis de entrada.

In [6]:
X = df[features].to_numpy()
y = df[target].to_numpy()

X_design = np.column_stack([np.ones(len(X)), X])

coeficientes, *_ = np.linalg.lstsq(X_design, y, rcond=None)

pred = X_design @ coeficientes
erro = y - pred

mae = np.mean(np.abs(erro))
rmse = np.sqrt(np.mean(erro ** 2))
r2 = 1 - np.sum(erro ** 2) / np.sum((y - y.mean()) ** 2)

tabela_metricas = pd.DataFrame({
    "métrica": ["MAE (p.p.)", "RMSE (p.p.)", "R²"],
    "valor": [mae, rmse, r2],
})

desvio_x = df[features].std(ddof=1).to_numpy()
desvio_y = df[target].std(ddof=1)
coeficientes_padronizados = coeficientes[1:] * desvio_x / desvio_y

tabela_coeficientes = pd.DataFrame({
    "variável": features,
    "coeficiente": coeficientes[1:],
    "coeficiente_padronizado": coeficientes_padronizados,
})
tabela_coeficientes["impacto_padronizado_absoluto"] = (
    tabela_coeficientes["coeficiente_padronizado"].abs()
)
tabela_coeficientes = tabela_coeficientes.sort_values(
    "impacto_padronizado_absoluto", ascending=False
).reset_index(drop=True)

display(tabela_metricas.round(3))
display(tabela_coeficientes.round(3))


,métrica,valor
0,MAE (p.p.),0.276
1,RMSE (p.p.),0.344
2,R²,0.714


,variável,coeficiente,coeficiente_padronizado,impacto_padronizado_absoluto
0,taxa_abandono_carrinho_pct,-0.060,-0.650,0.650
1,profundidade_scroll_pct,0.024,0.457,0.457
2,tempo_primeiro_clique_s,-0.094,-0.328,0.328


Interprete o erro do modelo em relação à taxa de conversão.

**Resposta:**

O **MAE de 0,276 ponto percentual** significa que, dentro destes 180 dias, a previsão se afasta da taxa de conversão observada em 0,276 p.p. em média. O **RMSE de 0,344 p.p.** penaliza mais os erros grandes; por isso seu valor superior ao MAE indica que alguns desvios maiores têm peso relevante. Como a conversão média é **5,869%**, MAE e RMSE correspondem a aproximadamente **4,70%** e **5,85%** dessa média. O modelo explica **71,4%** da variação observada (R² = **0,714**), mas essa avaliação é feita na mesma amostra usada no ajuste e pode ser otimista.

Os coeficientes têm sinais coerentes com a exploração: mantendo as demais entradas constantes, +1 p.p. de abandono está associado a **-0,060 p.p.** de conversão; +1 p.p. de scroll, a **+0,024 p.p.**; e +1 segundo até o primeiro clique, a **-0,094 p.p.**. Como as unidades são diferentes, a comparação adequada usa os coeficientes padronizados: abandono (**-0,650**), scroll (**+0,457**) e primeiro clique (**-0,328**), novamente com abandono em primeiro lugar por impacto absoluto.


## Parte 3: Análise de Sensibilidade

Calcule a sensibilidade para duas variáveis de entrada usando uma variação de 10%.

Use a fórmula: sensibilidade igual à variação percentual da saída dividida pela variação percentual da entrada.

In [7]:
def prever_linha(linha):
    entrada = np.array([1] + [linha[feature] for feature in features])
    return float(entrada @ coeficientes)


linha_base = df[features].mean().to_dict()
saida_base = prever_linha(linha_base)

linha_base, saida_base

({'taxa_abandono_carrinho_pct': 47.54587889307049,
  'profundidade_scroll_pct': 62.44918010647032,
  'tempo_primeiro_clique_s': 6.9708957453209095},
 5.868747841831934)

In [8]:
# Duas variáveis escolhidas com base na exploração da Parte 1.
variaveis_escolhidas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
]

if len(variaveis_escolhidas) != 2:
    raise ValueError("Preencha variaveis_escolhidas com duas variáveis da lista features.")

variaveis_invalidas = [v for v in variaveis_escolhidas if v not in features]

if variaveis_invalidas:
    raise ValueError(f"Variáveis fora de features: {variaveis_invalidas}")

variacao_entrada = 0.10


def calcular_sensibilidade(variavel):
    linha_cenario = linha_base.copy()
    valor_original = linha_base[variavel]
    valor_alterado = valor_original * (1 + variacao_entrada)
    linha_cenario[variavel] = valor_alterado

    saida_nova = prever_linha(linha_cenario)
    variacao_saida = (saida_nova - saida_base) / saida_base
    indice_sensibilidade = variacao_saida / variacao_entrada

    return {
        "variável": variavel,
        "valor_original": valor_original,
        "valor_alterado": valor_alterado,
        "saída_original": saida_base,
        "saída_nova": saida_nova,
        "variação_saída_pct": variacao_saida * 100,
        "índice_sensibilidade": indice_sensibilidade,
    }


resultados = [calcular_sensibilidade(v) for v in variaveis_escolhidas]
tabela_sensibilidade = pd.DataFrame(resultados)
display(tabela_sensibilidade.round(3))

# Ranking complementar das três entradas, calculado com a mesma perturbação de 10%.
ranking_sensibilidade = pd.DataFrame(
    [calcular_sensibilidade(v) for v in features]
)
ranking_sensibilidade["impacto_absoluto"] = (
    ranking_sensibilidade["índice_sensibilidade"].abs()
)
ranking_sensibilidade = ranking_sensibilidade.sort_values(
    "impacto_absoluto", ascending=False
).reset_index(drop=True)
display(
    ranking_sensibilidade[
        ["variável", "variação_saída_pct", "índice_sensibilidade", "impacto_absoluto"]
    ].round(3)
)


,variável,valor_original,valor_alterado,saída_original,saída_nova,variação_saída_pct,índice_sensibilidade
0,taxa_abandono_carrinho_pct,47.546,52.300,5.869,5.582,-4.891,-0.489
1,profundidade_scroll_pct,62.449,68.694,5.869,6.020,2.578,0.258


,variável,variação_saída_pct,índice_sensibilidade,impacto_absoluto
0,taxa_abandono_carrinho_pct,-4.891,-0.489,0.489
1,profundidade_scroll_pct,2.578,0.258,0.258
2,tempo_primeiro_clique_s,-1.118,-0.112,0.112


Compare os índices de sensibilidade e indique qual variável tem maior impacto sobre a taxa de conversão.

Mostre o raciocínio: cite os valores da tabela e explique o que eles significam para a decisão.

**Resposta:**

Ao elevar o abandono em 10%, de **47,546%** para **52,300%**, a conversão prevista cai de **5,869%** para **5,582%**: variação de **-4,891%** e índice de sensibilidade **-0,489**. Para o scroll, o aumento de 10%, de **62,449%** para **68,694%**, eleva a conversão a **6,020%**: variação de **+2,578%** e índice **+0,258**.

Logo, o abandono tem maior impacto: seu índice absoluto (**0,489**) é cerca de **1,90 vez** o do scroll (**0,258**). O ranking completo mantém a ordem abandono (**0,489**) > scroll (**0,258**) > primeiro clique (**0,112**). Os sinais e a ordem são coerentes com correlações e coeficientes padronizados; a concordância entre três leituras distintas reforça a prioridade do abandono, sem provar causalidade.


## Parte 4: Decisão

Recomende uma ação de produto ou interface com base na análise.

Sua recomendação deve citar os números da tabela de sensibilidade.

**Resposta:**

Aponte uma limitação, risco ou hipótese da sua análise.

**Resposta:**

### Recomendação

Priorizar um **experimento de interface voltado a reduzir o abandono do carrinho**, medindo seu efeito causal por teste A/B. A tabela mostra que um aumento de 10% no abandono, de **47,546%** para **52,300%**, reduz a conversão prevista de **5,869%** para **5,582%** (**-0,287 p.p.; -4,891%; índice -0,489**). Pela linearidade do modelo, uma redução relativa de 10% no abandono levaria ao efeito simétrico estimado de aproximadamente **+0,287 p.p.**, maior que o ganho de **+0,151 p.p.** obtido no cenário de +10% de scroll. O teste deve atacar fricções no fluxo do carrinho, mas o notebook não identifica qual componente específico causa o abandono; essa escolha exige diagnóstico adicional.

### Limitações, riscos e hipóteses

A base é simulada e contém apenas 180 observações diárias. O modelo é linear, aditivo e avaliado na própria amostra; portanto, seus resultados são associativos e podem não generalizar para usuários reais. A sensibilidade altera uma variável por vez, mantendo as demais fixas, e não representa interações. Além disso, a recomendação pressupõe que reduzir abandono seja causalmente capaz de elevar conversão, hipótese que precisa ser validada pelo experimento.


## Ao Além dos Aléns

Faça uma simulação de Monte Carlo para estimar como a taxa de conversão pode variar sob incerteza nas variáveis de entrada.

In [9]:
# Simulação de Monte Carlo com as distribuições e limites definidos na atividade.
n_simulacoes = 1000

amostras = pd.DataFrame({
    "taxa_abandono_carrinho_pct": rng.normal(
        linha_base["taxa_abandono_carrinho_pct"], 5, n_simulacoes
    ).clip(25, 75),
    "profundidade_scroll_pct": rng.normal(
        linha_base["profundidade_scroll_pct"], 8, n_simulacoes
    ).clip(25, 95),
    "tempo_primeiro_clique_s": rng.normal(
        linha_base["tempo_primeiro_clique_s"], 1.5, n_simulacoes
    ).clip(2, 15),
})

amostras_design = np.column_stack([
    np.ones(len(amostras)),
    amostras[features].to_numpy(),
])
previsoes = amostras_design @ coeficientes

# Cenário da recomendação: redução relativa de 10% no abandono, sob a mesma incerteza.
amostras_recomendacao = amostras.copy()
amostras_recomendacao["taxa_abandono_carrinho_pct"] = (
    amostras_recomendacao["taxa_abandono_carrinho_pct"]
    * (1 - variacao_entrada)
).clip(25, 75)

amostras_recomendacao_design = np.column_stack([
    np.ones(len(amostras_recomendacao)),
    amostras_recomendacao[features].to_numpy(),
])
previsoes_recomendacao = amostras_recomendacao_design @ coeficientes
ganho_recomendacao = previsoes_recomendacao - previsoes


def resumir_simulacao(valores, cenario):
    return {
        "cenário": cenario,
        "média": np.mean(valores),
        "desvio_padrão": np.std(valores, ddof=1),
        "P10": np.quantile(valores, 0.10),
        "P25": np.quantile(valores, 0.25),
        "P50": np.quantile(valores, 0.50),
        "P75": np.quantile(valores, 0.75),
        "P90": np.quantile(valores, 0.90),
    }


resumo_monte_carlo = pd.DataFrame([
    resumir_simulacao(previsoes, "Referência"),
    resumir_simulacao(previsoes_recomendacao, "Abandono -10%"),
])

resumo_risco = pd.DataFrame({
    "indicador": [
        "ganho médio (p.p.)",
        "P10 do ganho (p.p.)",
        "P90 do ganho (p.p.)",
        "P(conversão com ação < saída-base)",
    ],
    "valor": [
        np.mean(ganho_recomendacao),
        np.quantile(ganho_recomendacao, 0.10),
        np.quantile(ganho_recomendacao, 0.90),
        np.mean(previsoes_recomendacao < saida_base),
    ],
})

display(resumo_monte_carlo.round(3))
display(resumo_risco.round(3))


,cenário,média,desvio_padrão,P10,P25,P50,P75,P90
0,Referência,5.869,0.393,5.369,5.617,5.854,6.131,6.379
1,Abandono -10%,6.155,0.368,5.687,5.915,6.143,6.394,6.632


,indicador,valor
0,ganho médio (p.p.),0.286
1,P10 do ganho (p.p.),0.245
2,P90 do ganho (p.p.),0.325
3,P(conversão com ação < saída-base),0.212


In [10]:
df_monte_carlo = pd.concat([
    pd.DataFrame({
        "taxa_conversao_pct_prevista": previsoes,
        "cenário": "Referência",
    }),
    pd.DataFrame({
        "taxa_conversao_pct_prevista": previsoes_recomendacao,
        "cenário": "Abandono -10%",
    }),
], ignore_index=True)

fig = px.histogram(
    df_monte_carlo,
    x="taxa_conversao_pct_prevista",
    color="cenário",
    nbins=35,
    barmode="overlay",
    opacity=0.65,
    histnorm="probability density",
    title="Monte Carlo: conversão sob incerteza e cenário recomendado",
    labels={
        "taxa_conversao_pct_prevista": "Taxa de conversão prevista (%)",
        "probability density": "Densidade",
    },
)
fig.show()


Interprete o que a distribuição simulada indica sobre o risco da sua recomendação.

**Resposta:**

No cenário de referência, a conversão simulada tem média de **5,869%**, desvio-padrão de **0,393 p.p.** e intervalo central P10–P90 de **5,369% a 6,379%**. Portanto, mesmo sem mudar o produto, a incerteza conjunta das três entradas produz uma faixa relevante: 10% das simulações ficam abaixo de 5,369% e 10% acima de 6,379%.

Com redução de 10% no abandono, a média sobe para **6,155%** e o P10–P90 passa a **5,687%–6,632%**. O ganho médio é **0,286 p.p.**; comparando cada simulação sob as mesmas condições, seu P10–P90 é **0,245–0,325 p.p.**. Ainda assim, em **21,2%** das simulações a conversão após a ação fica abaixo da saída-base determinística de 5,869%, porque scroll e tempo de clique também variam. Isso não significa que a ação piorou esses casos — o efeito pareado do modelo continua positivo —, mas mostra que ela não elimina o risco operacional.

A simulação assume entradas normais independentes, dispersões fixas e coeficientes sem incerteza; além disso, não adiciona o erro residual do modelo. Assim, os percentis descrevem apenas o mecanismo simulado no notebook e podem subestimar a incerteza real. A recomendação deve ser validada por experimento antes de implantação ampla.


## Política de Uso de IA

O uso de IA é permitido para apoio técnico, revisão de texto e estudo dos conceitos.

As escolhas de variáveis, os cálculos, a comparação dos índices e a recomendação devem refletir sua análise dos resultados deste notebook.

Você deve ser capaz de explicar qualquer resposta entregue.

Respostas sem relação com os números gerados, com indícios de cópia ou que não possam ser justificadas poderão ser tratadas como fora da proposta.

## Instruções de entrega

A entrega deverá ser feita no GitHub ou no próprio Google Colab.

Links **sem permissão** de acesso terão um desconto de 20% na nota.